# Coastal Water Quality from Tanager Hyperspectral Data

**Scene:** `20250626_082641_74_4001` — southern Mozambique coast  
**Catalog:** [Tanager Open STAC](https://www.planet.com/data/stac/browser/tanager-core-imagery/catalog.json) → Coastal and Water Bodies

Maps water extent, turbidity, and chlorophyll proxies from Tanager-1 surface reflectance; compares full hyperspectral indices to a simulated 8-band workflow.

In [ ]:
from pathlib import Path
import sys

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))

import matplotlib.pyplot as plt
import numpy as np

from tanager_coastal.indices import compute_water_indices, multispectral_ndwi_approx
from tanager_coastal.stac import asset_href, get_item, list_collections
from tanager_coastal.viz import load_cloud_mask, load_cog, plot_rgb_and_indices

COLLECTION = 'coastal-water-bodies'
SCENE = '20250626_082641_74_4001'
DATA_DIR = ROOT / 'data'

## 1. Browse the Open STAC catalog

In [ ]:
for row in list_collections():
    print(f"{row['title']:30}  {row['id']}")

item = get_item(COLLECTION, SCENE)
print('\nSelected scene:', item['id'])
print('Acquired:', item['properties']['datetime'])
print('BBox:', item['bbox'])

## 2. Download assets (run once from repo root)

```bash
python scripts/download_scene.py          # RGB + cloud mask
python scripts/download_scene.py --full   # + ortho_sr_hdf5 cube
```

In [ ]:
rgb, _ = load_cog(next(DATA_DIR.glob('*_ortho_visual.tif')))
cloud_mask = load_cloud_mask(next(DATA_DIR.glob('*_ortho_beta_udm.tif')))

fig, ax = plt.subplots(figsize=(10, 8))
display = np.moveaxis(rgb[:3], 0, -1)
display = np.clip(display / np.percentile(display, 98), 0, 1)
ax.imshow(display)
ax.set_title('Tanager ortho_visual — southern Mozambique')
ax.axis('off')
plt.show()

## 3. Hyperspectral water quality indices

Using [HyperCoast](https://hypercoast.org/examples/tanager/) on `ortho_sr_hdf5` (~426 bands, 380–2500 nm).

In [ ]:
import hypercoast

sr_path = next(DATA_DIR.glob('*_ortho_sr_hdf5.h5'))
dataset = hypercoast.read_tanager(sr_path, stac_url=asset_href(item, 'ortho_sr_hdf5'))

reflectance = dataset['surface_reflectance'].values
wavelengths = dataset['wavelength'].values

indices = compute_water_indices(reflectance, wavelengths, nodata_mask=cloud_mask)
indices['ndwi_multispectral'] = multispectral_ndwi_approx(
    reflectance, wavelengths, nodata_mask=cloud_mask
)

plot_rgb_and_indices(
    rgb,
    indices,
    title=f'Coastal water quality — {SCENE}',
    output=ROOT / 'outputs' / 'water_quality_maps.png',
)

## 4. Hyperspectral vs. simulated 8-band NDWI

In [ ]:
diff = np.abs(indices['ndwi'] - indices['ndwi_multispectral'])
print(f'Mean absolute NDWI difference: {np.nanmean(diff):.4f}')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, arr, title in zip(
    axes,
    [indices['ndwi'], indices['ndwi_multispectral'], diff],
    ['Hyperspectral NDWI', 'Simulated 8-band NDWI', '|Difference|'],
):
    im = ax.imshow(arr, cmap='Blues' if 'Diff' not in title else 'magma')
    ax.set_title(title)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'ndwi_comparison.png', dpi=150, bbox_inches='tight')
plt.show()